# Silver Revenue Data

Promotes the daily revenue data from the Bronze layer (`workspace.bronze.revenue_data`) to the Silver layer with proper numeric typing, then derives business insights: overall KPIs, monthly trend, best days, and order-value efficiency.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import trim, col, when, coalesce, lit

## Reading from the Bronze Table

**Purpose:** Loads the raw daily revenue data (Date, Orders, Gross_sales, Returns, Coupons, Net_sales, Taxes, Shipping, Total_sales) that the Bronze layer ingested from `revenue.csv`.

In [0]:
df = spark.table('workspace.bronze.revenue_data')

## Data Transformations

**Purpose:** Cleans the bronze data and promotes it to the Silver layer.

**Steps:**
1. Convert `Date` from timestamp to date-only (`yyyy-mm-dd`), matching the format used in `workspace.silver.silver_orders_data`.
2. Strip stray quote characters from `Net_sales` and `Total_sales` — five rows carry a leading apostrophe from CSV escaping (e.g. `"'-27.34"`) — and cast them from string to `double` so they can be aggregated.
3. Overwrite `workspace.silver.silver_revenue_data` with the cleaned DataFrame.

**Note:** `Gross_sales`, `Returns`, `Coupons`, `Taxes`, and `Shipping` already arrive as doubles; only the two string money columns need casting.

In [0]:
# df.display()

# 1. Date - convert timestamp to date-only (yyyy-mm-dd),
#    matching the format used in workspace.silver.silver_orders_data
df = df.withColumn('Date', F.to_date(col('Date')))

# 2. Cast the string money columns to double so they can be aggregated.
#    Five rows carry a stray leading apostrophe from CSV escaping (e.g. "'-27.34"),
#    so strip quotes first, then cast.
df_clean = (df
    .withColumn('Net_sales', F.regexp_replace(col('Net_sales'), "['\"]", '').cast('double'))
    .withColumn('Total_sales', F.regexp_replace(col('Total_sales'), "['\"]", '').cast('double'))
)

df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.silver.silver_revenue_data')

df_clean.display()

## Revenue Insights

**Purpose:** Turns the cleaned silver data into business answers across four angles:

1. **Overall KPIs** — period totals and averages
2. **Monthly revenue trend** — how net sales evolve month over month
3. **Top 10 days by net sales** — the biggest revenue spikes
4. **Efficiency metrics** — average order value, daily rates, return and coupon impact

### 1. Overall KPIs

One-row summary of the full period: days covered, date range, total orders, gross sales, returns, coupons, net sales, shipping, and total collected.

In [0]:
silver_df = spark.table('workspace.silver.silver_revenue_data')

kpi_df = silver_df.agg(
    F.count('*').alias('total_days'),
    F.min('Date').alias('first_date'),
    F.max('Date').alias('last_date'),
    F.sum('Orders').alias('total_orders'),
    F.round(F.sum('Gross_sales'), 2).alias('total_gross_sales'),
    F.round(F.sum('Returns'), 2).alias('total_returns'),
    F.round(F.sum('Coupons'), 2).alias('total_coupons'),
    F.round(F.sum('Net_sales'), 2).alias('total_net_sales'),
    F.round(F.sum('Shipping'), 2).alias('total_shipping'),
    F.round(F.sum('Total_sales'), 2).alias('total_sales'),
)

display(kpi_df)

### 2. Monthly Revenue Trend

Net sales and order volume grouped by month, ordered chronologically — shows seasonality and growth across the covered period.

In [0]:
# Monthly net sales trend
monthly_df = (silver_df
    .withColumn('month', F.to_date(F.date_trunc('month', col('Date'))))
    .groupBy('month')
    .agg(
        F.sum('Orders').alias('orders'),
        F.round(F.sum('Net_sales'), 2).alias('net_sales'),
        F.round(F.avg('Net_sales'), 2).alias('avg_net_sales_per_day'),
    )
    .orderBy('month')
)

display(monthly_df)

### 3. Top 10 Days by Net Sales

The ten highest-grossing days with their order count and full revenue breakdown — useful for spotting campaign or seasonal spikes.

In [0]:
top_days_df = (silver_df
    .select('Date', 'Orders', 'Gross_sales', 'Returns', 'Coupons', 'Net_sales', 'Shipping', 'Total_sales')
    .orderBy(F.col('Net_sales').desc())
    .limit(10)
)

display(top_days_df)

### 4. Efficiency Metrics

Average order value (net sales / orders), daily averages, and the impact of returns, coupons, and shipping relative to gross sales.

In [0]:
# Efficiency: average order value, daily rates, and discount/return impact
efficiency_df = silver_df.agg(
    F.round(F.sum('Net_sales') / F.sum('Orders'), 2).alias('avg_order_value'),
    F.round(F.sum('Net_sales') / F.count('*'), 2).alias('avg_net_sales_per_day'),
    F.round(F.sum('Orders') / F.count('*'), 2).alias('avg_orders_per_day'),
    F.round(F.sum('Shipping') / F.sum('Net_sales') * 100, 2).alias('shipping_pct_of_net'),
    F.round(F.sum('Returns') / F.sum('Gross_sales') * 100, 2).alias('return_rate_pct'),
    F.round(F.sum('Coupons') / F.sum('Gross_sales') * 100, 2).alias('coupon_discount_pct'),
    F.sum(when(col('Orders') == 0, 1).otherwise(0)).alias('zero_sales_days'),
)

display(efficiency_df)